# Demostración del paquete `minigpt`

Ejemplo mínimo de funcionamiento del transformer causal desarrollado en
`minigpt/`, cargando los pesos preentrenados de GPT-2 (124 M) y generando
texto con **dos esquemas posicionales**:

* **Base** — incrustaciones posicionales absolutas aprendidas (réplica de GPT-2).
* **RoPE** — *Rotary Position Embeddings* (los mismos pesos, descartando la
  incrustación posicional `wpe`).

> La variante RoPE se incluye solo como demostración de la API: al reutilizar
> pesos entrenados con posiciones absolutas, su salida será incoherente.  Ese
> comportamiento es el esperado y está analizado en el cuaderno principal
> `PLN_practica_1_RoPE.ipynb`.

## 1. Configuración

Cargamos `tiktoken` (BPE de GPT-2) y comprobamos el dispositivo disponible.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import tiktoken
import minigpt as mg

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = tiktoken.get_encoding("gpt2")

print(f"minigpt {mg.__version__} | torch {torch.__version__}")
gpu = f" — {torch.cuda.get_device_name(0)}" if device == "cuda" else ""
print(f"dispositivo: {device}{gpu}")

/home/usuario/A4/PL/venv_PL/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


minigpt 1.0.0 | torch 2.6.0+cu124


dispositivo: cuda — NVIDIA GeForce RTX 4070 Laptop GPU


## 2. Carga de los modelos

`pretrained_gpt2()` descarga los pesos oficiales la primera vez (los guarda en
`gpt2_weights/`) y rellena nuestra implementación.  Pedimos las dos variantes
posicionales.

In [2]:
models = {
    "Base (pos. aprendidas)": mg.pretrained_gpt2(positional="learned", device=device),
    "RoPE": mg.pretrained_gpt2(positional="rope", device=device),
}
for name, model in models.items():
    print(f"{name:<28} {mg.format_parameters(mg.count_parameters(model)):>14} parámetros")

Cargando pesos de GPT-2 desde /home/usuario/A4/PL/Practica1_PLN1/gpt2_weights ...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:  16%|█▌        | 23/148 [00:00<00:00, 228.93it/s]

Loading weights:  42%|████▏     | 62/148 [00:00<00:00, 315.21it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 711.55it/s]

Cargando pesos de GPT-2 desde /home/usuario/A4/PL/Practica1_PLN1/gpt2_weights ...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6590.20it/s]

Base (pos. aprendidas)          163.037.184 parámetros
RoPE                            162.250.752 parámetros


## 3. Función auxiliar de generación

`mg.generate` implementa decodificación voraz (`temperature=None`) o muestreo
con temperatura y `top_k`. Encapsulamos el troceado y la decodificación en una
función cómoda.

In [3]:
@torch.no_grad()
def completar(model, prompt, max_new_tokens=30, temperature=None, top_k=None, seed=None):
    idx = torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=device)[None, :]
    out = mg.generate(
        model,
        idx,
        max_new_tokens=max_new_tokens,
        context_size=model.cfg.context_length,
        temperature=temperature,
        top_k=top_k,
        seed=seed,
    )
    return tokenizer.decode(out[0].tolist())

## 4. Ejemplos de generación (decodificación voraz)

Completamos varias frases de ejemplo con cada variante.  El modelo **Base**
debe producir texto coherente; la variante **RoPE** con pesos de GPT-2 hereda
un esquema posicional para el que sus pesos no fueron entrenados.

In [4]:
prompts = [
    "Every effort moves you",
    "The sky was clear and the stars",
    "In a small town by the sea,",
    "The secret to happiness is",
    "Artificial intelligence will",
]

for prompt in prompts:
    print("=" * 70)
    print(f"indicación: {prompt!r}")
    for name, model in models.items():
        print(f"  [{name}]")
        print(f"    {completar(model, prompt, max_new_tokens=30)}")
    print()

indicación: 'Every effort moves you'
  [Base (pos. aprendidas)]


    Every effort moves you forward.

The first step is to understand the importance of your work.

The second step is to understand the importance of your work.
  [RoPE]


    Every effort moves you every every every every.
I I I I I I am a good I am a little I am I I am a real I I I I

indicación: 'The sky was clear and the stars'
  [Base (pos. aprendidas)]


    The sky was clear and the stars were shining.

"I'm sorry, I'm sorry, I'm sorry," he said. "I'm sorry, I'm sorry.
  [RoPE]


    The sky was clear and the stars all all all all all all all all all all all all all all all all all all all all all all all all all all all all all all

indicación: 'In a small town by the sea,'
  [Base (pos. aprendidas)]


...

    In a small town by the sea, all all all all all all all all all all all all all all all all all all all all all all all all all all all all all all

indicación: 'The secret to happiness is'
  [Base (pos. aprendidas)]


    The secret to happiness is to be happy.

The secret to happiness is to be happy.

The secret to happiness is to be happy.

The secret
  [RoPE]


    The secret to happiness is the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the

indicación: 'Artificial intelligence will'
  [Base (pos. aprendidas)]


    Artificial intelligence will be able to do things like search for and find people, and to do things like find out who's in the right place at the right time.
  [RoPE]


    Artificial intelligence will work work work work work work work work work work work work work work work work work work work work work work work work work work work work work work



## 5. Ejemplo con muestreo

Sobre el modelo **Base**, comparamos decodificación voraz con muestreo
(top-k = 50) a dos temperaturas a partir de la misma indicación.

In [5]:
prompt = "Once upon a time"

print(f"indicación: {prompt!r}\n")
print("[voraz]")
print(" ", completar(models["Base (pos. aprendidas)"], prompt, max_new_tokens=40))
print("\n[muestreo, temperatura 0.7, top_k=50]")
print(" ", completar(models["Base (pos. aprendidas)"], prompt, max_new_tokens=40,
                      temperature=0.7, top_k=50, seed=7))
print("\n[muestreo, temperatura 1.0, top_k=50]")
print(" ", completar(models["Base (pos. aprendidas)"], prompt, max_new_tokens=40,
                      temperature=1.0, top_k=50, seed=7))

indicación: 'Once upon a time'

[voraz]


  Once upon a time, the world was a place of great beauty and great danger. The world was a place of great danger, and the world was a place of great danger. The world was a place of great danger

[muestreo, temperatura 0.7, top_k=50]


  Once upon a time, we have a group of people who are on the planet that are living on a planet that is totally uninhabitable. It's like being on Mount Everest.

We're living on the

[muestreo, temperatura 1.0, top_k=50]


  Once upon a time, these men were like a sea without a king.

When a man was king, and his army fell suddenly upon him and threatened him with death, they came to the aid of a man


## 6. Conclusión

* La implementación **Base** reproduce a GPT-2 y genera texto coherente.
* La misma arquitectura con **RoPE** (sin `wpe`) no funciona bien con pesos
  preentrenados con posiciones absolutas: RoPE **debe entrenarse desde cero o
  con fine-tuning**.  El cuaderno `PLN_practica_1_RoPE.ipynb` cuantifica este
  efecto (hipótesis H3).